# PRD-AGI Complete Ultimate Edition Setup

## ⚠️ Before running — Setup Colab Secrets first!

1. Click the 🔑 **Secrets** icon in the left sidebar
2. Add these secrets:
   - `GEMINI_API_KEY` → your Gemini API key (required)
   - `NGROK_AUTH_TOKEN` → from https://dashboard.ngrok.com/auth (required)
   - `GROQ_API_KEY` → from https://console.groq.com (optional, free)

## Steps:
1. Run Cell 1 (Drive mount)
2. Run Cell 2 (Install dependencies & Setup Repo — takes 3-5 minutes)
3. Run Cell 3 (Set API keys)
4. Run Cell 4 (Start all services)
5. Open the ngrok URL shown

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/prd_agi_data/uploads
!mkdir -p /content/drive/MyDrive/prd_agi_data/checkpoints

In [ ]:
# Install Node.js
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

# Ensure we are in the correct cloned repository
REPO_URL = "https://github.com/kkomyoeminaung/prd-agi-ultimate.git"
import os, subprocess
os.chdir('/content')
if not os.path.exists('/content/prd-agi-ultimate'):
    print("📥 Cloning repository...")
    result = subprocess.run(f"git clone {REPO_URL}", shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ Clone FAILED: {result.stderr}")
        print("💡 Alternative: Upload your zip manually and run:")
        print("   !unzip prd-agi-complete-ultimate-edition.zip -d /content/prd-agi-ultimate")
        raise SystemExit("Fix repo URL or upload zip manually")
    print("✅ Cloned successfully!")
else:
    print("🔄 Pulling latest updates...")
    os.chdir('/content/prd-agi-ultimate')
    os.system("git pull")

os.chdir('/content/prd-agi-ultimate')
print(f"📂 Working dir: {os.getcwd()}")

# Explicit installer steps
print("📦 Installing concurrently...")
os.system("npm install -g concurrently tsx 2>/dev/null || true")
print("📦 Installing backend-node deps...")
os.system("cd /content/prd-agi-ultimate/backend-node && npm install")
print("✅ All Node.js dependencies installed!")

# Install Python backend dependencies
if os.path.exists('prd-core/requirements.txt'):
    !pip install -r prd-core/requirements.txt
if os.path.exists('software-agents/requirements.txt'):
    !pip install -r software-agents/requirements.txt
if os.path.exists('private-llm/requirements.txt'):
    !pip install -r private-llm/requirements.txt

# Install React frontend dependencies
if os.path.exists('package.json'):
    !npm install


In [ ]:
# Set API keys from Colab secrets
import os
from google.colab import userdata

try:
   os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except:
   print('⚠️ GEMINI_API_KEY not found in Colab Secrets.')
try:
   os.environ['NGROK_AUTH_TOKEN'] = userdata.get('NGROK_AUTH_TOKEN')
except:
   print('⚠️ NGROK_AUTH_TOKEN not found in Colab Secrets.')
try:
   os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except:
   pass

os.environ['SQLITE_PATH'] = '/content/drive/MyDrive/prd_agi_data/prd_llm.db'
os.environ['UPLOAD_DIR'] = '/content/drive/MyDrive/prd_agi_data/uploads/'


In [ ]:
import os, sys, time, subprocess
from pyngrok import ngrok
import requests as req

print("📂 Finding project directory...")
target_dir = '/content/prd-agi-ultimate'
if os.path.exists(target_dir):
    os.chdir(target_dir)
    print(f"✅ Working in: {target_dir}")
else:
    print("❌ Repo not found! Run clone cell first."); raise SystemExit

print("🧹 Killing stale processes...")
os.system("killall -9 ngrok node vite python3 2>/dev/null || true")
time.sleep(2)

# Ngrok setup
!pip install pyngrok -q
print("🔗 Starting ngrok tunnel...")
try:
    ngrok.kill()
    token = os.environ.get('NGROK_AUTH_TOKEN', '')
    if not token:
        print("⚠️ NGROK_AUTH_TOKEN not set in Colab Secrets!")
        raise SystemExit("Add NGROK_AUTH_TOKEN to Colab Secrets")
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(3000).public_url
    print(f"\n{'='*50}")
    print(f"🚀 PRD-AGI LIVE at: {public_url}")
    print(f"{'='*50}\n")
except Exception as e:
    print(f"❌ Ngrok error: {e}")

# Start ALL services via colab_start.py ONLY (no double frontend)
print("⏳ Starting all services...")
proc = subprocess.Popen('python3 colab_start.py', shell=True)
print("✅ Services starting in background. Wait 10-15 seconds then open the URL.")

print("⏳ Waiting for services to be ready...")
services = {
    "Private LLM (8000)": "http://localhost:8000/",
    "PRD Core (8001)":    "http://localhost:8001/",
    "Agents (8002)":      "http://localhost:8002/",
    "Backend (3001)":     "http://localhost:3001/api/health",
}

for attempt in range(45):  # max 90 seconds
    all_up = True
    for name, url in services.items():
        try:
            r = req.get(url, timeout=1)
            print(f"  ✅ {name} — UP")
        except:
            all_up = False
    if all_up:
        print("\n🎉 All services ready! Open the URL above.")
        break
    time.sleep(2)
else:
    print("\n⚠️ Some services took too long. Check logs above.")
